In [26]:
import numpy as np
import pandas as pd
import textwrap
import nltk
from nltk import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer

In [27]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [28]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [29]:
df = pd.read_csv('bbc-text.csv')

In [30]:
df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [31]:
labels = set(df['category'])
labels

{'business', 'entertainment', 'politics', 'sport', 'tech'}

In [32]:
# Pick a label whose data we want to train from
label = 'business'

In [33]:
texts = df[df['category'] == label]['text']
texts.head()

,text
1,worldcom boss left books alone former worldc...
11,virgin blue shares plummet 20% shares in austr...
12,crude oil prices back above $50 cold weather a...
15,s korean credit card firm rescued south korea ...
18,japanese banking battle at an end japan s sumi...


In [34]:
# collect counts
probs = {} # key: (w(t-1), w(t+1)), value: {w(t): count(w(t))}

for doc in texts:
  lines = doc.split("\n")
  for line in lines:
    tokens = word_tokenize(line)
    for i in range(len(tokens) - 2):
      t_0 = tokens[i]
      t_1 = tokens[i + 1]
      t_2 = tokens[i + 2]
      key = (t_0, t_2)
      if key not in probs:
        probs[key] = {}

      # add count for middle token
      if t_1 not in probs[key]:
        probs[key][t_1] = 1
      else:
        probs[key][t_1] += 1

In [35]:
# normalize probabilities key is surrounding word and d dict
for key, d in probs.items():
  # d should represent a distribution
  total = sum(d.values())
  #k is middle word v is value
  for k, v in d.items():
    d[k] = v / total

In [36]:
probs

{('worldcom', 'left'): {'boss': 1.0},
 ('boss', 'books'): {'left': 1.0},
 ('left', 'alone'): {'books': 1.0},
 ('books', 'former'): {'alone': 1.0},
 ('alone', 'worldcom'): {'former': 1.0},
 ('former', 'boss'): {'worldcom': 0.625, 'yukos': 0.25, 'shell': 0.125},
 ('worldcom', 'bernie'): {'boss': 0.7142857142857143,
  'chief': 0.2857142857142857},
 ('boss', 'ebbers'): {'bernie': 0.8333333333333334,
  'bernard': 0.16666666666666666},
 ('bernie', 'who'): {'ebbers': 0.5, 'banton': 0.5},
 ('ebbers', 'is'): {'who': 0.5, 'trial': 0.5},
 ('who', 'accused'): {'is': 1.0},
 ('is', 'of'): {'accused': 0.08823529411764706,
  'part': 0.20588235294117646,
  'one': 0.47058823529411764,
  'because': 0.058823529411764705,
  'symbolic': 0.029411764705882353,
  'confident': 0.058823529411764705,
  'also': 0.029411764705882353,
  'tens': 0.029411764705882353,
  'hopeful': 0.029411764705882353},
 ('accused', 'overseeing'): {'of': 1.0},
 ('of', 'an'): {'overseeing': 0.5,
  'kfb': 0.125,
  'opening': 0.125,
  'a

In [37]:
texts.iloc[0].split("\n")

['worldcom boss  left books alone  former worldcom boss bernie ebbers  who is accused of overseeing an $11bn (£5.8bn) fraud  never made accounting decisions  a witness has told jurors.  david myers made the comments under questioning by defence lawyers who have been arguing that mr ebbers was not responsible for worldcom s problems. the phone company collapsed in 2002 and prosecutors claim that losses were hidden to protect the firm s shares. mr myers has already pleaded guilty to fraud and is assisting prosecutors.  on monday  defence lawyer reid weingarten tried to distance his client from the allegations. during cross examination  he asked mr myers if he ever knew mr ebbers  make an accounting decision  .  not that i am aware of   mr myers replied.  did you ever know mr ebbers to make an accounting entry into worldcom books   mr weingarten pressed.  no   replied the witness. mr myers has admitted that he ordered false accounting entries at the request of former worldcom chief financ

output[] empty list to store each spunned paragraph

In [38]:
def spin_document(doc):
  # split the document into lines (paragraphs)
  lines = doc.split("\n")
  output = []
  for line in lines:
    if line:
      new_line = spin_line(line)
    else:
      new_line = line
    output.append(new_line)
  return "\n".join(output)

In [39]:
detokenizer = TreebankWordDetokenizer()

In [40]:
texts.iloc[0].split("\n")[0]

'worldcom boss  left books alone  former worldcom boss bernie ebbers  who is accused of overseeing an $11bn (£5.8bn) fraud  never made accounting decisions  a witness has told jurors.  david myers made the comments under questioning by defence lawyers who have been arguing that mr ebbers was not responsible for worldcom s problems. the phone company collapsed in 2002 and prosecutors claim that losses were hidden to protect the firm s shares. mr myers has already pleaded guilty to fraud and is assisting prosecutors.  on monday  defence lawyer reid weingarten tried to distance his client from the allegations. during cross examination  he asked mr myers if he ever knew mr ebbers  make an accounting decision  .  not that i am aware of   mr myers replied.  did you ever know mr ebbers to make an accounting entry into worldcom books   mr weingarten pressed.  no   replied the witness. mr myers has admitted that he ordered false accounting entries at the request of former worldcom chief financi

In [41]:
detokenizer.detokenize(word_tokenize(texts.iloc[0].split("\n")[0]))

'worldcom boss left books alone former worldcom boss bernie ebbers who is accused of overseeing an $11bn (£5.8bn) fraud never made accounting decisions a witness has told jurors . david myers made the comments under questioning by defence lawyers who have been arguing that mr ebbers was not responsible for worldcom s problems . the phone company collapsed in 2002 and prosecutors claim that losses were hidden to protect the firm s shares . mr myers has already pleaded guilty to fraud and is assisting prosecutors . on monday defence lawyer reid weingarten tried to distance his client from the allegations . during cross examination he asked mr myers if he ever knew mr ebbers make an accounting decision . not that i am aware of mr myers replied . did you ever know mr ebbers to make an accounting entry into worldcom books mr weingarten pressed . no replied the witness . mr myers has admitted that he ordered false accounting entries at the request of former worldcom chief financial officer s

In [42]:
def sample_word(d):
  p0 = np.random.random()
  cumulative = 0
  for t, p in d.items():
    cumulative += p
    if p0 < cumulative:
      return t
  assert(False) # should never get here

In [43]:
def spin_line(line):
  tokens = word_tokenize(line)
  i = 0
  output = [tokens[0]]
  while i < (len(tokens) - 2):
    t_0 = tokens[i]
    t_1 = tokens[i + 1]
    t_2 = tokens[i + 2]
    key = (t_0, t_2)
    p_dist = probs[key]
    if len(p_dist) > 1 and np.random.random() < 0.3:
      # let's replace the middle word
      middle = sample_word(p_dist)
      output.append(t_1)
      output.append("<" + middle + ">")
      output.append(t_2)

      # we won't replace the 3rd token since the middle
      # token was dependent on it
      # instead, skip ahead 2 steps
      i += 2
    else:
      # we won't replace this middle word
      output.append(t_1)
      i += 1
  # append the final token - only if there was no replacement
  if i == len(tokens) - 2:
    output.append(tokens[-1])
  return detokenizer.detokenize(output)

In [44]:
np.random.seed(1234)

In [45]:
i = np.random.choice(texts.shape[0])
doc = texts.iloc[i]
new_doc = spin_document(doc)

In [46]:
print(textwrap.fill(
    new_doc, replace_whitespace=False, fix_sentence_endings=True))

honda wins china copyright ruling japan s honda has won a copyright
case in beijing further evidence that china is taking <eyeing> a
tougher line on protecting intellectual property rights <market>. a
<the> court ruled that chongqing lifan industry group must stop
selling honda brand motorbikes and said it must <to> pay 1.47m yuan
($177 600) in compensation <november>. internationally recognized
regulation is now <primarily> a key part of china <china> s plans for
developing its economy analysts said . beijing also has been
threatened <engaged> with sanctions if it fails to clamp down .
chinese firms copy products ranging from computer software and spark
plugs to baby milk and compact discs . despite the fact <moment> that
product piracy is a major problem foreign companies have only
occasionally won cases and the compensation awarded has usually <not>
been small . still recent rulings <rulings> and announcements <ntpc>
will have boosted optimism that attitudes are changing . earlier
<